# Day 6 — Pipeline Integration and Drift Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-06-pipeline-integration.ipynb)

**Badge:** Review  
**Course:** Great Expectations for Data Quality

---

## What you will learn

- How GX integrates with orchestration tools like Apache Airflow and Prefect
- How to build a four-step pipeline (ingest → clean → aggregate → load) with per-stage Checkpoints
- How to design schema-drift Expectations and detect column additions/removals
- How to write a drift report function comparing observed columns against the stored suite contract

> **Tip:** Run the Checkpoint at the *start* of each stage (validating input), not the end. This ensures a failing Checkpoint aborts the stage before any side effects occur — the 'validate inputs, not outputs' pattern.

## Reading: GX + Airflow Integration

Full guide: [How to use Great Expectations with Airflow](https://docs.greatexpectations.io/docs/deployment_patterns/how_to_use_great_expectations_with_airflow)

The integration pattern in Airflow looks like this:

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
import great_expectations as gx

def validate_ingest(**context):
    gx_context = gx.get_context()
    checkpoint = gx_context.checkpoints.get('ingest_checkpoint')
    result = checkpoint.run()
    if not result.success:
        raise ValueError('GX validation failed — aborting DAG')

with DAG('orders_pipeline', ...) as dag:
    validate_task = PythonOperator(
        task_id='validate_ingest',
        python_callable=validate_ingest,
    )
```

Key Airflow integration principles:
- Each DAG task maps to one pipeline stage.
- The validation task comes *before* the transformation task in the dependency chain.
- Raising an exception in the validation task marks the DAG run as failed and halts downstream tasks.

## Reading: GX + Prefect Integration

Full guide: [How to use Great Expectations in Prefect flows](https://docs.greatexpectations.io/docs/deployment_patterns/how_to_use_great_expectations_in_prefect_flows)

In Prefect, a GX Checkpoint call wraps naturally into a `@task`:

```python
from prefect import flow, task
import great_expectations as gx

@task
def validate_stage(stage_name: str, df):
    ctx = gx.get_context()
    checkpoint = ctx.checkpoints.get(f'{stage_name}_checkpoint')
    result = checkpoint.run(batch_parameters={'dataframe': df})
    if not result.success:
        raise ValueError(f'{stage_name} validation failed')
    return df

@flow
def orders_pipeline(raw_df):
    clean_df = validate_stage('ingest', raw_df)
    agg_df   = validate_stage('clean', clean_df)
    ...
```

Prefect surfaces task failures in its UI with a red state, and retries can be configured at the task level independently of validation logic.

## Install dependencies

In [ ]:
%pip install great_expectations --quiet

## Setup: Context and Data

We simulate a four-step pipeline with synthetic event data: ingest raw events, clean nulls and types, aggregate per-user, then load the aggregated result.

In [ ]:
import great_expectations as gx
import pandas as pd
import numpy as np
import sys
from datetime import datetime, timedelta

context = gx.get_context()

# ---- Synthetic raw events data ----
rng = np.random.default_rng(7)
N = 300

base_ts = datetime(2024, 1, 1)
raw_df = pd.DataFrame({
    'event_id':  [f'EVT-{i:06d}' for i in range(1, N + 1)],
    'user_id':   rng.integers(100, 199, size=N),
    'event_ts':  [base_ts + timedelta(hours=int(h)) for h in rng.integers(0, 720, size=N)],
    'event_type': rng.choice(['click', 'view', 'purchase', 'login'], size=N),
    'value':     rng.uniform(0.0, 250.0, size=N).round(2),
})

print('Raw data shape:', raw_df.shape)
print('Columns:', list(raw_df.columns))
raw_df.head(3)

## 1. Schema-Drift Expectations

Schema drift is one of the most common data quality failures in long-running pipelines. The pattern:
1. Define expectations that lock in the expected column set.
2. Run them at the ingest stage before any transformation.
3. If a column is renamed or dropped upstream, the Checkpoint fires immediately.

Key expectations for schema drift:
- `ExpectTableColumnsToMatchSet` — exact column set (order-insensitive).
- `ExpectColumnToExist` — individual critical columns.
- `ExpectTableColumnCountToEqual` — absolute column count guard.

In [ ]:
# ---- Ingest schema suite ----
EXPECTED_COLUMNS = {'event_id', 'user_id', 'event_ts', 'event_type', 'value'}

suite_ingest = context.suites.add(gx.ExpectationSuite(name='ingest.schema'))

# Exact column set (order-insensitive)
suite_ingest.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=list(EXPECTED_COLUMNS),
        exact_match=True,
    )
)

# Critical individual columns
for col in ['user_id', 'event_ts']:
    suite_ingest.add_expectation(gx.expectations.ExpectColumnToExist(column=col))

# No nulls on critical columns
for col in ['event_id', 'user_id', 'event_ts']:
    suite_ingest.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))

context.suites.save(suite_ingest)
print('ingest.schema suite:', len(suite_ingest.expectations), 'expectations')

## 2. Drift Report Function

Beyond the binary pass/fail, it is useful to print a human-readable report of what changed. The drift report compares the current batch's columns against the column set stored in the suite's `ExpectTableColumnsToMatchSet` kwargs.

In [ ]:
def drift_report(df: pd.DataFrame, suite: gx.ExpectationSuite) -> dict:
    """Compare df columns against the column set in the suite.

    Looks for an ExpectTableColumnsToMatchSet expectation in the suite
    and reports added/removed columns.

    Returns:
        dict with keys 'expected', 'observed', 'added', 'removed'.
    """
    expected_cols = set()
    for exp in suite.expectations:
        if type(exp).__name__ == 'ExpectTableColumnsToMatchSet':
            expected_cols = set(exp.column_set)
            break

    observed_cols = set(df.columns)
    added   = observed_cols - expected_cols
    removed = expected_cols - observed_cols

    report = {
        'expected': sorted(expected_cols),
        'observed': sorted(observed_cols),
        'added':    sorted(added),
        'removed':  sorted(removed),
    }

    print(f'Expected columns : {report["expected"]}')
    print(f'Observed columns : {report["observed"]}')
    if added:
        print(f'  DRIFT — added   : {report["added"]}')
    if removed:
        print(f'  DRIFT — removed : {report["removed"]}')
    if not added and not removed:
        print('  No drift detected')

    return report


# Run on clean data — should show no drift
print('=== Clean data drift report ===')
drift_report(raw_df, context.suites.get('ingest.schema'))

## 3. Simulating Schema Drift

We remove the `event_ts` column to simulate an upstream schema change, then re-run the Checkpoint and drift report.

In [ ]:
# Register data source + batch definition
ds_ingest   = context.data_sources.add_pandas('ingest_source')
asset_ingest = ds_ingest.add_dataframe_asset('ingest_asset')
batch_def_ingest = asset_ingest.add_batch_definition_whole_dataframe('ingest_batch')

vd_ingest = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='vd_ingest_schema',
        data=batch_def_ingest,
        suite=context.suites.get('ingest.schema'),
    )
)

cp_ingest = context.checkpoints.add(
    gx.Checkpoint(
        name='ingest_checkpoint',
        validation_definitions=[vd_ingest],
        result_format={'result_format': 'COMPLETE'},
    )
)

# --- Simulate drift: drop event_ts ---
drifted_df = raw_df.drop(columns=['event_ts'])
print('Drifted data columns:', list(drifted_df.columns))
print()

print('=== Drifted data drift report ===')
drift_report(drifted_df, context.suites.get('ingest.schema'))
print()

# Run checkpoint on drifted data
drift_cp_result = cp_ingest.run(batch_parameters={'dataframe': drifted_df})
print('Checkpoint success on drifted data:', drift_cp_result.success)
for identifier, vr in drift_cp_result.run_results.items():
    for r in vr.results:
        if not r.success:
            print(f'  FAIL: {r.expectation_config.type} | {r.expectation_config.kwargs}')

## 4. Four-Step Pipeline with Per-Stage Checkpoints

Now we build the full pipeline. Each stage:
1. **Validates its input** using a Checkpoint.
2. **Transforms** the data.
3. **Passes** the result to the next stage.

If any Checkpoint fails, a `RuntimeError` is raised and the pipeline halts before side effects.

Stages:
- **Ingest:** Raw CSV / API → raw DataFrame. Validate schema.
- **Clean:** Drop nulls, cast types. Validate no remaining nulls in critical columns.
- **Aggregate:** Group by `user_id`, sum `value`. Validate row count and value range.
- **Load:** Write to destination (simulated). Validate final column set.

In [ ]:
# ---- Suite definitions for each stage ----

def _save_suite(name: str, expectations: list) -> gx.ExpectationSuite:
    suite = context.suites.add(gx.ExpectationSuite(name=name))
    for exp in expectations:
        suite.add_expectation(exp)
    context.suites.save(suite)
    return suite

# Clean stage: no nulls on key columns, event_type in known set
suite_clean = _save_suite('clean.schema', [
    gx.expectations.ExpectColumnValuesToNotBeNull(column='user_id'),
    gx.expectations.ExpectColumnValuesToNotBeNull(column='event_ts'),
    gx.expectations.ExpectColumnValuesToNotBeNull(column='value'),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='event_type', value_set=['click', 'view', 'purchase', 'login']
    ),
])

# Aggregate stage: reasonable row count, non-negative sums
suite_agg = _save_suite('aggregate.schema', [
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=1000),
    gx.expectations.ExpectColumnValuesToBeBetween(column='value', min_value=0.0),
    gx.expectations.ExpectColumnToExist(column='user_id'),
    gx.expectations.ExpectColumnToExist(column='value'),
])

# Load stage: required output columns
suite_load = _save_suite('load.schema', [
    gx.expectations.ExpectTableColumnsToMatchSet(
        column_set=['user_id', 'value', 'event_count'], exact_match=True
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(column='user_id'),
])

print('All pipeline suites created')
for s in ['ingest.schema', 'clean.schema', 'aggregate.schema', 'load.schema']:
    suite = context.suites.get(s)
    print(f'  {s}: {len(suite.expectations)} expectations')

In [ ]:
# ---- Build per-stage checkpoints ----

def make_stage_checkpoint(stage: str, suite_name: str) -> gx.Checkpoint:
    """Create a DataSource + BatchDef + ValidationDef + Checkpoint for one pipeline stage."""
    ds   = context.data_sources.add_pandas(f'{stage}_source')
    asset = ds.add_dataframe_asset(f'{stage}_asset')
    bd   = asset.add_batch_definition_whole_dataframe(f'{stage}_batch')
    vd   = context.validation_definitions.add(
        gx.ValidationDefinition(
            name=f'vd_{stage}',
            data=bd,
            suite=context.suites.get(suite_name),
        )
    )
    cp = context.checkpoints.add(
        gx.Checkpoint(
            name=f'{stage}_cp',
            validation_definitions=[vd],
            result_format={'result_format': 'SUMMARY'},
        )
    )
    return cp


cp_clean = make_stage_checkpoint('clean', 'clean.schema')
cp_agg   = make_stage_checkpoint('aggregate', 'aggregate.schema')
cp_load  = make_stage_checkpoint('load', 'load.schema')

print('Per-stage checkpoints ready: ingest_cp, clean_cp, aggregate_cp, load_cp')

In [ ]:
# ---- Pipeline implementation ----

def gate(cp: gx.Checkpoint, df: pd.DataFrame, stage: str) -> pd.DataFrame:
    """Run a checkpoint; raise RuntimeError if validation fails."""
    result = cp.run(batch_parameters={'dataframe': df})
    status = 'PASS' if result.success else 'FAIL'
    print(f'  [{stage}] validation: {status}')
    if not result.success:
        failed = [
            r.expectation_config.type
            for _, vr in result.run_results.items()
            for r in vr.results if not r.success
        ]
        raise RuntimeError(f'{stage} validation FAILED: {failed}')
    return df


def run_pipeline(raw: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """Four-step pipeline with validate-before-transform at each stage."""
    print('=== Pipeline run ===')

    # INGEST: validate raw schema
    gate(cp_ingest, raw, 'ingest')
    if verbose:
        print(f'  [ingest] rows: {len(raw)}')

    # CLEAN: drop rows with nulls in critical columns, cast value to float
    clean = raw.dropna(subset=['user_id', 'event_ts', 'value']).copy()
    clean['value'] = clean['value'].astype(float)
    gate(cp_clean, clean, 'clean')
    if verbose:
        print(f'  [clean] rows: {len(clean)}')

    # AGGREGATE: sum value per user
    agg = (
        clean.groupby('user_id', as_index=False)
        .agg(value=('value', 'sum'), event_count=('event_id', 'count'))
    )
    gate(cp_agg, agg, 'aggregate')
    if verbose:
        print(f'  [aggregate] rows: {len(agg)}')

    # LOAD: final column selection
    loaded = agg[['user_id', 'value', 'event_count']].copy()
    gate(cp_load, loaded, 'load')
    if verbose:
        print(f'  [load] rows: {len(loaded)}')

    print('=== Pipeline complete ===')
    return loaded


# Run on clean data
output = run_pipeline(raw_df)
print()
print('Output sample:')
print(output.head())

## 5. Simulating Pipeline Failure via Schema Drift

Remove `event_ts` from the raw data and confirm the ingest gate fires before the clean stage runs.

In [ ]:
drifted_raw = raw_df.drop(columns=['event_ts'])
print('Running pipeline on drifted data (missing event_ts)...')

try:
    run_pipeline(drifted_raw, verbose=False)
except RuntimeError as exc:
    print(f'Pipeline halted at ingest: {exc}')
    print()
    print('Drift report:')
    drift_report(drifted_raw, context.suites.get('ingest.schema'))

## Challenge — Extend the Pipeline with a New Stage

1. Add a **filter stage** between clean and aggregate that keeps only `event_type == 'purchase'`.
2. Write a `filter.schema` suite that validates: at least 1 row remains, `event_type` column exists, and all values equal `'purchase'`.
3. Wire it into `run_pipeline()` with a gate.
4. Confirm the pipeline still runs end-to-end on the clean synthetic data.
5. (Bonus) Inject an upstream issue that causes the filter stage to produce 0 rows and confirm the gate catches it.

In [ ]:
# ---- Challenge solution ----

# 1 & 2. Define filter.schema suite
suite_filter = _save_suite('filter.schema', [
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=1_000_000),
    gx.expectations.ExpectColumnToExist(column='event_type'),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='event_type', value_set=['purchase']
    ),
])
print('filter.schema suite:', len(suite_filter.expectations), 'expectations')

# 3. Checkpoint for filter stage
cp_filter = make_stage_checkpoint('filter', 'filter.schema')


def run_pipeline_with_filter(raw: pd.DataFrame) -> pd.DataFrame:
    print('=== Extended Pipeline run ===')

    gate(cp_ingest, raw, 'ingest')
    clean = raw.dropna(subset=['user_id', 'event_ts', 'value']).copy()
    clean['value'] = clean['value'].astype(float)
    gate(cp_clean, clean, 'clean')

    # Filter stage — purchases only
    filtered = clean[clean['event_type'] == 'purchase'].copy()
    gate(cp_filter, filtered, 'filter')
    print(f'  [filter] rows: {len(filtered)}')

    agg = (
        filtered.groupby('user_id', as_index=False)
        .agg(value=('value', 'sum'), event_count=('event_id', 'count'))
    )
    gate(cp_agg, agg, 'aggregate')
    loaded = agg[['user_id', 'value', 'event_count']].copy()
    gate(cp_load, loaded, 'load')

    print('=== Extended Pipeline complete ===')
    return loaded


# 4. Run on clean data
output2 = run_pipeline_with_filter(raw_df)
print(f'Output rows: {len(output2)}')

# 5. Bonus: inject data with no purchases
print()
print('Testing with no-purchase data...')
no_purchase_df = raw_df.copy()
no_purchase_df['event_type'] = 'click'  # all clicks — filter produces 0 rows

try:
    run_pipeline_with_filter(no_purchase_df)
except RuntimeError as exc:
    print(f'Gate caught zero-row filter: {exc}')

## Recap

| Pattern | Implementation |
|---|---|
| Validate inputs, not outputs | Checkpoint runs *before* the transformation function |
| Schema drift detection | `ExpectTableColumnsToMatchSet(exact_match=True)` |
| Drift report | Compare `set(df.columns)` against suite kwargs |
| Airflow integration | `PythonOperator` wrapping `checkpoint.run()` + `raise` on failure |
| Prefect integration | `@task` wrapping `checkpoint.run()` + raise on failure |
| Pipeline halt | Raise `RuntimeError` (or `sys.exit(1)` in scripts) in the gate |

**Key architectural insight:** Each pipeline stage has a *contract* (its input suite). When upstream systems change, the contract breaks loudly at the boundary — not silently inside a transformation function hours later.

**Next up — Day 7:** Writing Custom Expectations and the full Capstone project.